# V1-07: 기업명 문자열 유사도 후보 검색

별칭 사전을 크게 만들지 않고, 오타·부분 표현에서 올바른 기업이 후보 상위권에 들어오는지 확인합니다.

- A. 정규화 인덱스만 사용: 정확히 일치하는 정규화 이름만 후보로 반환
- B. 문자열 유사도 fallback: 정확 후보가 없을 때 유사한 정규화 이름의 상위 후보를 반환

이번 notebook은 **후보 생성**만 평가합니다. 문자열 유사도 점수만으로 기업을 자동 확정하지 않습니다.

## 평가 기준

- **Candidate Recall@1**: 최상위 후보가 정답 기업인 비율
- **Candidate Recall@3**: 상위 3개 후보 안에 정답 기업이 있는 비율
- **p50/p95 지연시간**: 기업명 후보를 반환하는 데 걸린 시간
- 존재하지 않는 기업명은 어떤 후보와 유사하게 보이는지 점수만 확인합니다. 자동 선택 안전성은 다음 단계에서 threshold·점수 차이·사용자 확인 정책과 함께 평가합니다.

In [1]:
import io
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

from rapidfuzz import fuzz, process
import rapidfuzz

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

In [3]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes

print(f'RapidFuzz 버전: {rapidfuzz.__version__}')

RapidFuzz 버전: 3.14.5


In [4]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
print(f'기업코드 레코드 수: {len(corp_records):,}')

기업코드 레코드 수: 118,831


## 1. 정규화 이름 인덱스

이전 단계에서 채택한 인덱스를 후보 이름 목록으로 사용합니다. 이번에는 별칭 사전을 비워 두어 문자열 유사도 자체가 처리할 수 있는 범위를 확인합니다.

In [5]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(records_by_code.values()) for key, records_by_code in index.items()}

name_index = build_name_index(corp_records)
name_keys = list(name_index)
print(f'정규화 이름 키 수: {len(name_keys):,}')

정규화 이름 키 수: 218,777


## 2. 개발용 입력 사례

오타·부분 표현·한글/영문 표현을 함께 넣습니다. `네이버`, `삼전`은 문자열 유사도만으로 해결되는지 확인하기 위한 실패 가능 사례이며, 정답을 상위 후보에 반드시 넣어야 한다는 뜻은 아닙니다.

In [6]:
TEST_CASES = [
    {'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930'},
    {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '정답 종목코드': '005930'},
    {'유형': '한 글자 오타', '질문 기업명': 'SK하이닉수', '정답 종목코드': '000660'},
    {'유형': '부분 표현', '질문 기업명': '하이닉스', '정답 종목코드': '000660'},
    {'유형': '부분 표현', '질문 기업명': '현대자동', '정답 종목코드': '005380'},
    {'유형': '영문 대소문자', '질문 기업명': 'naver', '정답 종목코드': '035420'},
    {'유형': '한글 별칭', '질문 기업명': '네이버', '정답 종목코드': '035420'},
    {'유형': '짧은 약칭', '질문 기업명': '삼전', '정답 종목코드': '005930'},
    {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None},
]

TEST_CASES

[{'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930'},
 {'유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '정답 종목코드': '005930'},
 {'유형': '한 글자 오타', '질문 기업명': 'SK하이닉수', '정답 종목코드': '000660'},
 {'유형': '부분 표현', '질문 기업명': '하이닉스', '정답 종목코드': '000660'},
 {'유형': '부분 표현', '질문 기업명': '현대자동', '정답 종목코드': '005380'},
 {'유형': '영문 대소문자', '질문 기업명': 'naver', '정답 종목코드': '035420'},
 {'유형': '한글 별칭', '질문 기업명': '네이버', '정답 종목코드': '035420'},
 {'유형': '짧은 약칭', '질문 기업명': '삼전', '정답 종목코드': '005930'},
 {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None}]

## 3. 후보 생성 방식

A는 정확 인덱스 후보만 반환합니다. B는 정확 후보가 없을 때 RapidFuzz의 문자 유사도 점수로 상위 정규화 이름 20개를 찾고, 중복된 법인 후보를 합쳐 상위 3개를 반환합니다.

In [7]:
FUZZY_KEY_LIMIT = 20
RETURN_K = 3

def rank_exact_records(records):
    return [(record, 100.0) for record in sorted(
        records,
        key=lambda record: (bool(record['stock_code'].strip()), record['modify_date']),
        reverse=True,
    )]

def index_only_candidates(query):
    return rank_exact_records(name_index.get(normalize_name(query), []))[:RETURN_K], 'exact'

def fuzzy_fallback_candidates(query):
    exact_records = name_index.get(normalize_name(query), [])
    if exact_records:
        return rank_exact_records(exact_records)[:RETURN_K], 'exact'

    scored_keys = process.extract(
        normalize_name(query),
        name_keys,
        scorer=fuzz.ratio,
        limit=FUZZY_KEY_LIMIT,
    )
    best_by_corp_code = {}
    for key, score, _ in scored_keys:
        for record in name_index[key]:
            current = best_by_corp_code.get(record['corp_code'])
            if current is None or score > current[1]:
                best_by_corp_code[record['corp_code']] = (record, score)

    ranked = sorted(
        best_by_corp_code.values(),
        key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']),
        reverse=True,
    )
    return ranked[:RETURN_K], 'fuzzy'

## 4. Candidate Recall@K와 지연시간 비교

각 입력을 warm-up 2회 후 10회 반복합니다. 출력에서는 후보 법인명·종목코드·유사도 점수를 직접 확인합니다.

In [8]:
WARMUP_ITERATIONS = 2
MEASURE_ITERATIONS = 10

def evaluate_strategy(strategy_name, candidate_function):
    results = []
    latencies = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            candidate_function(case['질문 기업명'])

        case_latencies = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates, source = candidate_function(case['질문 기업명'])
            case_latencies.append((time.perf_counter() - started_at) * 1_000)

        candidate_stock_codes = [record['stock_code'] for record, _ in candidates]
        results.append({
            **case,
            '전략': strategy_name,
            '후보 출처': source,
            '후보': [
                {'법인명': record['corp_name'], '종목코드': record['stock_code'], '점수': round(score, 1)}
                for record, score in candidates
            ],
            'Recall@1': candidate_stock_codes[:1] == [case['정답 종목코드']] if case['정답 종목코드'] else None,
            'Recall@3': case['정답 종목코드'] in candidate_stock_codes[:RETURN_K] if case['정답 종목코드'] else None,
            '반복 중앙 지연시간(ms)': median(case_latencies),
        })
        latencies.extend(case_latencies)
    return results, latencies

index_results, index_latencies = evaluate_strategy('정규화 인덱스만', index_only_candidates)
fuzzy_results, fuzzy_latencies = evaluate_strategy('정규화 인덱스 + 문자열 유사도', fuzzy_fallback_candidates)

for row in index_results + fuzzy_results:
    print({key: row[key] for key in ['전략', '유형', '질문 기업명', '후보 출처', '후보', 'Recall@1', 'Recall@3']})

{'전략': '정규화 인덱스만', '유형': '정확 법인명', '질문 기업명': '삼성전자', '후보 출처': 'exact', '후보': [{'법인명': '삼성전자', '종목코드': '005930', '점수': 100.0}], 'Recall@1': True, 'Recall@3': True}
{'전략': '정규화 인덱스만', '유형': '한 글자 오타', '질문 기업명': '삼성전쟈', '후보 출처': 'exact', '후보': [], 'Recall@1': False, 'Recall@3': False}
{'전략': '정규화 인덱스만', '유형': '한 글자 오타', '질문 기업명': 'SK하이닉수', '후보 출처': 'exact', '후보': [], 'Recall@1': False, 'Recall@3': False}
{'전략': '정규화 인덱스만', '유형': '부분 표현', '질문 기업명': '하이닉스', '후보 출처': 'exact', '후보': [], 'Recall@1': False, 'Recall@3': False}
{'전략': '정규화 인덱스만', '유형': '부분 표현', '질문 기업명': '현대자동', '후보 출처': 'exact', '후보': [], 'Recall@1': False, 'Recall@3': False}
{'전략': '정규화 인덱스만', '유형': '영문 대소문자', '질문 기업명': 'naver', '후보 출처': 'exact', '후보': [{'법인명': 'NAVER', '종목코드': '035420', '점수': 100.0}], 'Recall@1': True, 'Recall@3': True}
{'전략': '정규화 인덱스만', '유형': '한글 별칭', '질문 기업명': '네이버', '후보 출처': 'exact', '후보': [], 'Recall@1': False, 'Recall@3': False}
{'전략': '정규화 인덱스만', '유형': '짧은 약칭', '질문 기업명': '삼전', '후보 출처': 'exact', '후보': [{

In [9]:
def percentile(values, percentile_value):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * percentile_value)]

def summarize_results(results, latencies):
    positive_rows = [row for row in results if row['정답 종목코드'] is not None]
    return {
        '전략': results[0]['전략'],
        'Candidate Recall@1': sum(row['Recall@1'] for row in positive_rows) / len(positive_rows),
        'Candidate Recall@3': sum(row['Recall@3'] for row in positive_rows) / len(positive_rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': percentile(latencies, 0.95),
    }

for summary in [summarize_results(index_results, index_latencies), summarize_results(fuzzy_results, fuzzy_latencies)]:
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in summary.items()})

{'전략': '정규화 인덱스만', 'Candidate Recall@1': 0.25, 'Candidate Recall@3': 0.25, 'p50 지연시간(ms)': 0.0033, 'p95 지연시간(ms)': 0.0061}
{'전략': '정규화 인덱스 + 문자열 유사도', 'Candidate Recall@1': 0.625, 'Candidate Recall@3': 0.75, 'p50 지연시간(ms)': 26.1142, 'p95 지연시간(ms)': 27.5321}


## 5. 실행 후 해석

- 문자열 유사도 후보의 Recall@3가 높아지면, 오타·부분 표현에서 사용자에게 보여 줄 후보를 넓힐 수 있습니다.
- `네이버`, `삼전`처럼 한글 별칭과 약칭이 상위 후보에 없으면 문자열 유사도만으로는 해결하기 어렵다는 근거가 됩니다.
- 존재하지 않는 기업명에도 후보가 반환될 수 있으므로, 이번 결과만으로 자동 선택 정책을 만들지 않습니다. 다음 단계에서 점수 threshold, 1위·2위 점수 차이, 상장 여부, 사용자 확인을 비교합니다.
- 개발 사례로 만든 결과는 최종 채택 근거가 아닙니다. holdout 사례를 추가한 뒤 후보 생성 품질과 자동 선택 안전성을 분리 평가합니다.